In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import scipy
import pandas as pd
import tifffile as tif #211108 update, the latest skimage does not have external anymore
import seaborn as sns
from scipy.stats import lognorm, nbinom
from PIL import Image
import skimage
import cv2
from scipy.ndimage import maximum_filter, label
from scipy.optimize import curve_fit
from scipy.io import savemat
from skimage import img_as_ubyte

In [2]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['xtick.major.size'] = 3
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['xtick.bottom'] = True
plt.rcParams['ytick.left'] = True
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['legend.edgecolor'] = 'w'


In [5]:
# function 6
import numpy as np
from scipy.ndimage import maximum_filter, label, generate_binary_structure
from scipy.spatial.distance import cdist
from scipy.ndimage import measurements

global max_exclude
max_exclude = 20
class MaximumFinder:
    
    def __init__(self, tolerance=10, strict=False, exclude_on_edges=False, light_background=False, min_distance=5):
        self.tolerance = tolerance
        self.strict = strict
        self.exclude_on_edges = exclude_on_edges
        self.light_background = light_background
        self.min_distance = min_distance  # Minimum allowed distance between maxima

    def find_maxima(self, image):
        """
        Finds the maxima in the given 3D image (z, y, x format).
        :param image: 3D numpy array representing the image (z, y, x).
        :return: List of maxima coordinates [(z1, y1, x1), (z2, y2, x2), ...]
        """
        if self.light_background:
            image = -image  # Invert image for minima detection
        
        # Apply maximum filter once and reuse the result
        max_filtered = maximum_filter(image, size=3)
        
        # Use a maximum filter to find local maxima
        local_max = (image == max_filtered) & (image > 0) # Strict local maxima detection
        
        # Apply adaptive threshold for prominence
        adaptive_threshold = max_filtered - self.tolerance
        threshold_mask = image >= adaptive_threshold
        local_max &= threshold_mask
        
        # # Exclude maxima on edges only in x and y dimensions (considering up to 10 pixels)
        # if self.exclude_on_edges:
        #     # max_exclude = 10
        #     exclude_range = 10
        #     for axis in [1, 2]:  # Apply only to y and x dimensions
        #         # exclude_range = min(max_exclude, image.shape[axis])
        #         local_max[(slice(None),) * axis + (slice(0, exclude_range),)] = False
        #         local_max[(slice(None),) * axis + (slice(-exclude_range, None),)] = False
        # if self.exclude_on_edges:
        #     # max_exclude = 20
        #     exclude_range_z = min(max_exclude, image.shape[0])
        #     exclude_range_y = min(max_exclude, image.shape[1])
        #     exclude_range_x = min(max_exclude, image.shape[2])
        
        #     # Exclude top and bottom rows
        #     local_max[:, :exclude_range_y, :] = False
        #     local_max[:, -exclude_range_y:, :] = False
        
        #     # Exclude left and right columns
        #     local_max[:, :, :exclude_range_x] = False
        #     local_max[:, :, -exclude_range_x:] = False

        #     # Exclude front and back columns
        #     local_max[:exclude_range_z, :, :] = False
        #     local_max[-exclude_range_z:, :, :] = False




        
        # Label connected components of maxima
        labeled, num_features = label(local_max)
        
        # Extract maxima coordinates
        maxima_coords = np.column_stack(np.where(labeled > 0))
        
        # Apply optimized flood fill filtering
        maxima_coords = self._optimized_flood_fill_filter(maxima_coords, image)
        
        return [tuple(coord) for coord in maxima_coords]
    
    def _optimized_flood_fill_filter(self, maxima_coords, image):
        """
        Optimized flood fill using connected-component labeling.
        """
        scratch_image = np.copy(image)  # Temporary image for flood fill
        filled = np.zeros_like(image, dtype=bool)
        final_maxima = []
        
        for coord in maxima_coords[np.argsort(-image[tuple(maxima_coords.T)])]:  # Process highest first
            if filled[tuple(coord)]:
                continue  # Skip if already part of another filled area
            
            flood_region = self._perform_fast_flood_fill(scratch_image, coord)
            if np.any(filled[flood_region]):
                continue  # Discard if region overlaps another filled area
            
            # Keep the highest pixel in the flood-filled region
            highest_point = max(np.argwhere(flood_region), key=lambda p: image[tuple(p)])
            final_maxima.append(tuple(highest_point))
            filled[flood_region] = True

        for coord in final_maxima:
            if coord[1] < max_exclude or coord[1] >= image.shape[1] - max_exclude or coord[2] < max_exclude or coord[2] >= image.shape[2] - max_exclude:
                print("Edge Maximum Detected:", coord)

        if self.exclude_on_edges:
            final_maxima = [coord for coord in final_maxima if 
                            (coord[0] >= max_exclude and coord[0] < image.shape[0] - max_exclude) and 
                            (coord[1] >= max_exclude and coord[1] < image.shape[1] - max_exclude) and 
                            (coord[2] >= max_exclude and coord[2] < image.shape[2] - max_exclude)]
            print("Edge Maximum Removed")
        return np.array(final_maxima)
    
    def _perform_fast_flood_fill(self, image, start_coord):
        """
        Fast flood fill using connected-component labeling instead of stack-based expansion.
        """
        flood_region = np.zeros_like(image, dtype=bool)
        mask = np.abs(image - image[tuple(start_coord)]) <= self.tolerance
        labeled, num_labels = label(mask, structure=generate_binary_structure(3, 3))
        target_label = labeled[tuple(start_coord)]
        flood_region = labeled == target_label
        
        return flood_region



In [9]:
# input is the 3D voxal form of the SREV configuration
SREV_image = tif.imread('Nucleosome_Gaussian_rad5.tif')
print(SREV_image.std())
# CTRL_image_padded = np.concatenate((np.zeros(CTRL_image[np.newaxis, :].shape),CTRL_image[np.newaxis, :], np.zeros(CTRL_image[np.newaxis, :].shape)), axis = 0 )
# using function 6
max_finder = MaximumFinder(tolerance=1 * SREV_image.std(), exclude_on_edges=True, min_distance=15)
maxima = max_finder.find_maxima(SREV_image)
print("# maxima:",len(maxima))
print("Found maxima:", maxima)

slice_idx = 201
slice_img = SREV_image[slice_idx].copy()

# Convert to color image for visualization
output_img = cv2.cvtColor(100*slice_img, cv2.COLOR_GRAY2BGR)
# output_img = slice_img
for (z, y, x) in maxima:
    if z == slice_idx:  # Show only maxima in the selected slice
        cv2.circle(output_img, (x, y), 5, (0, 0, 255), -1)  # Red dot for maxima

# resized_image = cv2.resize(output_img, (900, 900))
# Show image
cv2.imshow(f'Maxima in Slice {slice_idx}', output_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

0.0024176938
Edge Maximum Detected: (425, 0, 0)
Edge Maximum Detected: (94, 599, 0)
Edge Maximum Detected: (599, 337, 599)
Edge Maximum Detected: (259, 0, 599)
Edge Maximum Detected: (427, 3, 0)
Edge Maximum Detected: (448, 0, 599)
Edge Maximum Detected: (0, 599, 482)
Edge Maximum Detected: (176, 0, 0)
Edge Maximum Detected: (599, 599, 132)
Edge Maximum Detected: (394, 0, 599)
Edge Maximum Detected: (0, 0, 519)
Edge Maximum Detected: (599, 0, 438)
Edge Maximum Detected: (0, 0, 268)
Edge Maximum Detected: (241, 0, 0)
Edge Maximum Detected: (281, 0, 599)
Edge Maximum Detected: (574, 599, 229)
Edge Maximum Detected: (0, 314, 599)
Edge Maximum Detected: (599, 599, 193)
Edge Maximum Detected: (469, 0, 599)
Edge Maximum Detected: (534, 0, 0)
Edge Maximum Detected: (599, 398, 0)
Edge Maximum Detected: (0, 391, 0)
Edge Maximum Detected: (599, 599, 162)
Edge Maximum Detected: (153, 337, 599)
Edge Maximum Detected: (599, 433, 0)
Edge Maximum Detected: (290, 0, 0)
Edge Maximum Detected: (367, 0, 

In [15]:
# SREV_image = tif.imread('D:\\20240904_SREV_EdU_BrdU_coords\\Simulated_EdUBrdU_Localizations\\Nucleosome\\OriginalConfiguration\\Nucleosome_washed_2nm_stack_Gaussian_rad5.tif')
np.save('Nucleosome_maxima (domain center)_github.npy', maxima)